In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q torch_geometric
!pip install -q class_resolver
!pip3 install pymatting

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 5.3 MB/s eta 0:00:00


In [3]:
# -*- coding: utf-8 -*-
"""
PneumoniaMNIST + ResNet18 features + AE-GCN_Recon
Merged implementation: ResNet18 extraction from PneumoniaMNIST script,
AE-GCN_Recon model/training (GCN encoder + adjacency reconstruction + classifier).
"""

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

from torch.utils.data import TensorDataset, DataLoader, Subset
from torchvision import models
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
)

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ============================================================
# Load PneumoniaMNIST 224x224
# ============================================================
data = np.load(
    "/content/drive/MyDrive/TejaswiAbburi_va797/Dataset/Medmnist_data/pneumoniamnist_224.npz",
    allow_pickle=True,
)

all_images = np.concatenate(
    [data["train_images"], data["val_images"], data["test_images"]], axis=0
)
all_labels = np.concatenate(
    [data["train_labels"], data["val_labels"], data["test_labels"]], axis=0
).squeeze()

images = all_images.astype(np.float32) / 255.0
images = np.repeat(images[:, None, :, :], 3, axis=1)  # (N,3,224,224)

X = torch.tensor(images)
y = torch.tensor(all_labels).long()

print("Images, labels shapes:", X.shape, y.shape)

# ============================================================
# Balanced subset (same as PneumoniaMNIST ARMA script)
# ============================================================
dataset = TensorDataset(X, y)

class0_indices = [i for i in range(len(y)) if y[i] == 0]
class1_indices = [i for i in range(len(y)) if y[i] == 1]

random.seed(SEED)
sampled_class0 = random.sample(class0_indices, min(2000, len(class0_indices)))
sampled_class1 = random.sample(class1_indices, min(2000, len(class1_indices)))

combined_indices = sampled_class0 + sampled_class1
random.shuffle(combined_indices)

final_dataset = Subset(dataset, combined_indices)
final_loader = DataLoader(final_dataset, batch_size=64, shuffle=False)

# ============================================================
# Extract ResNet18 features
# ============================================================
resnet = models.resnet18(pretrained=True)
resnet.fc = nn.Identity()
resnet = resnet.to(device)
resnet.eval()

resnet_feats, y_list = [], []

with torch.no_grad():
    for imgs, lbls in final_loader:
        imgs = imgs.to(device)
        feats = resnet(imgs)
        resnet_feats.append(feats.cpu())
        y_list.extend(lbls.cpu().tolist())

features = torch.cat(resnet_feats, dim=0).numpy().astype(np.float32)
y_labels = np.array(y_list).astype(np.int64)

num_nodes, num_feats = features.shape
print(f"Extracted ResNet18 Features: {features.shape}, Labels: {y_labels.shape}")

# ============================================================
# Graph construction
# ============================================================
ALPHA = 0.9  # from PneumoniaMNIST ARMA script


def create_adj(F_data, alpha=0.9):
    norms = np.linalg.norm(F_data, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    F_norm = F_data / norms
    W = np.dot(F_norm, F_norm.T)
    W = np.where(W >= alpha, 1.0, 0.0).astype(np.float32)
    if W.max() > 0:
        W = W / W.max()
    return W


def load_data(adj, node_feats):
    node_feats = torch.from_numpy(node_feats).float()
    rows, cols = np.nonzero(adj > 0)
    edge_index = torch.from_numpy(np.vstack([rows, cols]).astype(np.int64))
    return Data(x=node_feats, edge_index=edge_index)


W = create_adj(features, alpha=ALPHA)
np.fill_diagonal(W, 1.0)

data_graph = load_data(W, features).to(device)
print(data_graph)

# ============================================================
# AE-GCN model (from AE_GCN_Recon notebook)
# ============================================================
class GCN_AE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_nodes, device,
                 activ="RELU", dropout=0.25):
        super(GCN_AE, self).__init__()
        self.device = device
        self.dropout = dropout
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.conv3 = GCNConv(hidden_dim, num_nodes)

        activations = {
            "SELU": F.selu,
            "SiLU": F.silu,
            "GELU": F.gelu,
            "RELU": F.relu,
            "ELU": F.elu,
        }
        self.act = activations.get(activ, F.relu)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h1 = self.act(h)
        logits = self.conv2(h1, edge_index)
        h2 = torch.sigmoid(logits)
        A_hat = self.conv3(h1, edge_index)
        A_pred = torch.sigmoid(A_hat)
        return h2, A_pred, h1


def auto_loss(A_pred: torch.Tensor, A_tilde: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    if A_pred.dim() not in (2, 3):
        raise ValueError("A_pred must be shape (n,n) or (B,n,n)")
    if A_pred.shape != A_tilde.shape:
        raise ValueError("A_pred and A_tilde must have the same shape")
    if A_pred.dim() == 2:
        n = A_pred.size(0)
        D = A_tilde.sum(dim=1)
        D_inv_sqrt = torch.rsqrt(D + eps)
        D_inv_sqrt_mat = torch.diag(D_inv_sqrt)
        S = D_inv_sqrt_mat @ A_tilde @ D_inv_sqrt_mat
        A_pred_clamped = A_pred.clamp(min=eps, max=1.0)
        loss = - (S * torch.log(A_pred_clamped)).mean()
        return loss
    else:
        B, n, _ = A_pred.shape
        D = A_tilde.sum(dim=2)
        D_inv_sqrt = torch.rsqrt(D + eps)
        D_inv_sqrt_mat = torch.diag_embed(D_inv_sqrt)
        S = D_inv_sqrt_mat.bmm(A_tilde).bmm(D_inv_sqrt_mat)
        A_pred_clamped = A_pred.clamp(min=eps, max=1.0)
        loss = - (S * torch.log(A_pred_clamped)).mean()
        return loss


# ============================================================
# Hyperparameters (from AE_GCN_Recon notebook)
# ============================================================
feats_dim = num_feats
hidden_dim = 256
num_classes = 2
num_epochs = 2000
lr = 1e-4
weight_decay = 1e-4
batch_print_freq = 100
lambda_recon = 0.05   # as in AE-GCN notebook

OUTER_TEST_SIZE = 0.9
INNER_TEST_SIZE = 0.9   # from PneumoniaMNIST ARMA protocol

# ============================================================
# Cross-validation
# ============================================================
sss = StratifiedShuffleSplit(
    n_splits=20,
    test_size=OUTER_TEST_SIZE,
    random_state=SEED,
)

accuracies, precisions, recalls, f1_scores, aucs, ce_losses = [], [], [], [], [], []

y_all = y_labels.copy()

for fold, (train_val_idx, test_idx_global) in enumerate(
    sss.split(features, y_all), start=1
):
    print(f"\n=== Fold {fold} ===")

    control_idx = np.where(y_all == 0)[0]
    patient_idx = np.where(y_all == 1)[0]

    sss_class = StratifiedShuffleSplit(
        n_splits=20,
        test_size=INNER_TEST_SIZE,
        random_state=fold,
    )

    control_train_idx, _ = next(
        sss_class.split(features[control_idx], y_all[control_idx])
    )
    patient_train_idx, _ = next(
        sss_class.split(features[patient_idx], y_all[patient_idx])
    )

    control_train = control_idx[control_train_idx]
    patient_train = patient_idx[patient_train_idx]

    train_idx_final = np.concatenate([control_train, patient_train])
    np.random.shuffle(train_idx_final)

    print(f"Train Class0: {len(control_train)}, Train Class1: {len(patient_train)}")

    train_idx_t = torch.from_numpy(train_idx_final).long().to(device)
    y_train_tensor = torch.from_numpy(y_all[train_idx_final]).long().to(device)

    model = GCN_AE(
        feats_dim,
        hidden_dim,
        num_classes,
        num_nodes,          # for conv3 output size
        device,
        activ="RELU",
        dropout=0.25,
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    ce_loss_fn = nn.CrossEntropyLoss()
    A_tilde = torch.from_numpy(W).float().to(device)

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        # forward returns (node embeddings for classification, predicted adj, hidden)
        logits, A_pred, _ = model(data_graph.x, data_graph.edge_index)

        loss_sup = ce_loss_fn(logits[train_idx_t], y_train_tensor)
        loss_recon = auto_loss(A_pred, A_tilde)
        total_loss = loss_sup + lambda_recon * loss_recon

        if torch.isnan(total_loss) or torch.isinf(total_loss):
            print(
                f"NaN/Inf detected at fold {fold} epoch {epoch}. Aborting epoch."
            )
            break

        total_loss.backward()
        optimizer.step()

        if epoch % batch_print_freq == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                logits_eval, _, _ = model(
                    data_graph.x, data_graph.edge_index
                )
                preds_train = logits_eval[train_idx_t].argmax(dim=1)
                acc_train = accuracy_score(
                    y_train_tensor.cpu(), preds_train.cpu()
                )

            print(
                f"Fold {fold} Epoch {epoch}: "
                f"TotalLoss={total_loss.item():.6f} | "
                f"Sup={loss_sup.item():.6f} | "
                f"Recon={loss_recon.item():.6f} | "
                f"TrainAcc={acc_train:.4f}"
            )

    # Evaluation
    model.eval()
    with torch.no_grad():
        out_logits, _, _ = model(data_graph.x, data_graph.edge_index)
        preds = out_logits.argmax(dim=1).cpu().numpy()
        probs = F.softmax(out_logits, dim=1)[:, 1].cpu().numpy()

    y_test = y_all[test_idx_global]
    y_pred_test = preds[test_idx_global]
    y_prob_test = probs[test_idx_global]

    acc = accuracy_score(y_test, y_pred_test)
    prec = precision_score(y_test, y_pred_test, zero_division=0)
    rec = recall_score(y_test, y_pred_test, zero_division=0)
    f1 = f1_score(y_test, y_pred_test, zero_division=0)

    try:
        auc = roc_auc_score(y_test, y_prob_test)
    except ValueError:
        auc = float("nan")

    ce = log_loss(y_test, y_prob_test)

    accuracies.append(acc)
    precisions.append(prec)
    recalls.append(rec)
    f1_scores.append(f1)
    aucs.append(auc)
    ce_losses.append(ce)

    print(
        f"Fold {fold} → Acc={acc:.4f} | Prec={prec:.4f} | "
        f"Rec={rec:.4f} | F1={f1:.4f} | AUC={auc:.4f} | CE Loss={ce:.4f}"
    )

# ============================================================
# Average results
# ============================================================
print("\n=== Average Results Across 20 Folds ===")
print(f"Accuracy:  {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"Precision: {np.mean(precisions):.4f} ± {np.std(precisions):.4f}")
print(f"Recall:    {np.mean(recalls):.4f} ± {np.std(recalls):.4f}")
print(f"F1-score:  {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"AUC:       {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")
print(f"CE Loss:   {np.mean(ce_losses):.4f} ± {np.std(ce_losses):.4f}")

Device: cuda
Images, labels shapes: torch.Size([5856, 3, 224, 224]) torch.Size([5856])


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 192MB/s]


Extracted ResNet18 Features: (3583, 512), Labels: (3583,)
Data(x=[3583, 512], edge_index=[2, 1642013])

=== Fold 1 ===
Train Class0: 158, Train Class1: 200
Fold 1 Epoch 1: TotalLoss=0.691359 | Sup=0.691350 | Recon=0.000178 | TrainAcc=0.4413
Fold 1 Epoch 100: TotalLoss=0.434193 | Sup=0.434184 | Recon=0.000177 | TrainAcc=0.9078
Fold 1 Epoch 200: TotalLoss=0.414770 | Sup=0.414761 | Recon=0.000176 | TrainAcc=0.9078
Fold 1 Epoch 300: TotalLoss=0.408501 | Sup=0.408492 | Recon=0.000176 | TrainAcc=0.9134
Fold 1 Epoch 400: TotalLoss=0.405351 | Sup=0.405342 | Recon=0.000176 | TrainAcc=0.9162
Fold 1 Epoch 500: TotalLoss=0.403188 | Sup=0.403180 | Recon=0.000176 | TrainAcc=0.9190
Fold 1 Epoch 600: TotalLoss=0.401539 | Sup=0.401530 | Recon=0.000176 | TrainAcc=0.9190
Fold 1 Epoch 700: TotalLoss=0.400156 | Sup=0.400147 | Recon=0.000176 | TrainAcc=0.9218
Fold 1 Epoch 800: TotalLoss=0.398928 | Sup=0.398919 | Recon=0.000176 | TrainAcc=0.9218
Fold 1 Epoch 900: TotalLoss=0.397864 | Sup=0.397856 | Recon=0.0